<a href="https://colab.research.google.com/github/ritashreemukherjee123/Project-1-Applied-Search-Intelligence-Google-Search-Ranking-Discoverability.-/blob/main/work/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, numpy as np, pandas as pd
from sklearn.model_selection import KFold, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

SEED = 0
PATH = "features.parquet"
if not os.path.exists(PATH):
    from google.colab import drive; drive.mount("/content/drive")
    PATH = "/content/drive/MyDrive/colab_outputs/features.parquet"
df = pd.read_parquet(PATH)
df["as_of_date"] = pd.to_datetime(df["as_of_date"])

# A: does the label's base rate move with the calendar?
print(df.groupby("as_of_date")["is_declining"].agg(rows="size", base_rate="mean").round(3))

# B: does a date-proxy feature rank pages pooled but not within a date?
def pooled_vs_within(col, label="is_declining"):
    pooled = roc_auc_score(df[label], df[col].fillna(-1))
    within = [roc_auc_score(g[label], g[col].fillna(-1))
              for _, g in df.groupby("as_of_date") if g[label].nunique() == 2]
    return round(pooled, 3), round(float(np.mean(within)), 3)
if "active_days_90d" in df:
    print("active_days_90d pooled AUC, mean within-date AUC:", pooled_vs_within("active_days_90d"))


Mounted at /content/drive
             rows  base_rate
as_of_date                  
2026-06-01  17992      0.065
2026-06-15  62550      0.585
2026-06-29  76236      0.990
active_days_90d pooled AUC, mean within-date AUC: (np.float64(0.881), 0.515)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

MIN_CLICKS, RATIO = 10, 0.7
LAST_DATA = pd.Timestamp("2026-06-30")
as_of = pd.Timestamp("2026-06-15")
past_days = 15
future_days = (LAST_DATA - as_of).days            # 06-16..06-30 = 15 days
DROP = ["as_of_date","client_hash_id","content_hash_id","is_declining","clicks_future30","y_rate"]

d = df[(df["as_of_date"] == as_of) & (df["clicks_last30"] >= MIN_CLICKS)].copy().reset_index(drop=True)
d["y_rate"] = ((d["clicks_future30"]/future_days) < RATIO*(d["clicks_last30"]/past_days)).astype(int)
X = d.drop(columns=DROP).astype(float)
X = X.loc[:, X.notna().any()]                      # drops all-NaN columns such as impressions_prev30
y, g = d["y_rate"], d["client_hash_id"]
print(len(d), "rows | base rate", round(y.mean(), 3), "| features", X.shape[1])

def make_rf():
    return make_pipeline(SimpleImputer(strategy="constant", fill_value=-1),
        RandomForestClassifier(300, min_samples_leaf=20, n_jobs=-1, random_state=SEED))

def p_top(y_true, score, frac=0.05):
    k = max(1, int(len(y_true)*frac))
    return np.asarray(y_true)[np.argsort(-np.asarray(score), kind="stable")[:k]].mean()

def evaluate(name, splits, X, y):
    rows = []
    for tr, te in splits:
        p = make_rf().fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
        yt = y.iloc[te]
        rows.append(dict(auc=roc_auc_score(yt, p),
                         base_auc=roc_auc_score(yt, -X.iloc[te]["impressions_last30"]),
                         top5=p_top(yt, p), base=yt.mean()))
    r = pd.DataFrame(rows)
    return dict(split=name, folds=len(r), model_AUC=r.auc.mean(), sd=r.auc.std(),
                impressions_rule_AUC=r.base_auc.mean(), model_top5_precision=r.top5.mean(),
                base_rate=r.base.mean(), naive_AUC=0.5)

def time_splits(dates, embargo_days=30, min_train_dates=2):
    dv = pd.to_datetime(dates).values
    for T in np.unique(dv):
        tr = np.where(dv <= T - np.timedelta64(embargo_days, "D"))[0]
        te = np.where(dv == T)[0]
        if len(te) and len(np.unique(dv[tr])) >= min_train_dates:
            yield tr, te

res = [evaluate("random 5-fold (before)", KFold(5, shuffle=True, random_state=SEED).split(X), X, y),
       evaluate("grouped by client (after)", GroupKFold(5).split(X, y, g), X, y)]
# On the refreshed multi-date table, add:
#   evaluate("time-aware, 30d embargo", time_splits(d["as_of_date"]), X, y)
print(pd.DataFrame(res).round(3).to_string(index=False))

10436 rows | base rate 0.529 | features 43
                    split  folds  model_AUC    sd  impressions_rule_AUC  model_top5_precision  base_rate  naive_AUC
   random 5-fold (before)      5      0.700 0.014                 0.583                 0.890      0.529        0.5
grouped by client (after)      5      0.588 0.036                 0.584                 0.681      0.529        0.5


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FEATURES = list(X.columns)
bad = [c for c in FEATURES if c.endswith("_hash_id") or "future" in c.lower() or c in ("is_declining","y_rate")]
assert not bad, f"label / future / id columns in the features: {bad}"
FLAG_HINTS = ["provider","model_used","optimi","is_published","is_deleted","updated","score_"]
print("possible product flags (review by hand):", [c for c in FEATURES if any(h in c.lower() for h in FLAG_HINTS)])
print("exact-duplicate columns:", list(X.columns[X.T.duplicated(keep="first").values]))

grouped = list(GroupKFold(5).split(X, y, g))
honest = evaluate("grouped", grouped, X, y)["model_AUC"]

FAMILY = [c for c in X.columns if c.startswith(("clicks_","ctr_","has_clicks_","has_ctr_"))]
no_fam = evaluate("no family", grouped, X.drop(columns=FAMILY), y)["model_AUC"]
print(f"with clicks family {honest:.3f} | without ({len(FAMILY)} cols) {no_fam:.3f} | naive 0.500 | base rate {y.mean():.3f}")

leak = d["clicks_future30"] / np.maximum(d["clicks_last30"], 1)
leaky = evaluate("leak", grouped, X.assign(leak_future_ratio=leak.values), y)["model_AUC"]
print(f"deliberate leak: {honest:.3f} -> {leaky:.3f} (must jump toward 1.0, else the harness is broken)")

from sklearn.inspection import permutation_importance
imps = []
for tr, te in grouped:
    m = make_rf().fit(X.iloc[tr], y.iloc[tr])
    imps.append(permutation_importance(m, X.iloc[te], y.iloc[te], scoring="roc_auc",
                                       n_repeats=3, random_state=SEED, n_jobs=-1).importances_mean)
print(pd.Series(np.mean(imps, axis=0), index=X.columns).sort_values(ascending=False).head(6).round(4))
print("TOO GOOD - investigate" if honest > 0.85 else f"honest AUC {honest:.3f}: not suspiciously high")

possible product flags (review by hand): []
exact-duplicate columns: ['impressions_last30', 'impression_ratio_last_vs_prev30', 'has_competition', 'has_cpc', 'has_keyword_char_count', 'has_keyword_token_count', 'has_url_char_count', 'has_avg_position_last30', 'has_ctr_last30', 'has_ctr_prev30', 'has_impression_ratio_last_vs_prev30', 'has_content_created_date', 'content_type_comparison article']
with clicks family 0.588 | without (5 cols) 0.589 | naive 0.500 | base rate 0.529
deliberate leak: 0.588 -> 1.000 (must jump toward 1.0, else the harness is broken)
avg_position_last30    0.0323
impressions_90d        0.0183
impressions_last30     0.0159
ga4_sessions_last30    0.0097
ctr_last30             0.0083
keyword_token_count    0.0051
dtype: float64
honest AUC 0.588: not suspiciously high


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*  

**On one decision date (2026-06-15), across the clients with at least 100 pages, pages with average position worse than 12 declined more often than the date’s base rate. A single client supplies most of the top-ranked pages, and the within-client signal is weak but mostly positive (position AUC 0.50–0.60 in 13 of 15 clients). This is an observed, directional pattern from one date, usable as decision support for triage, not as a tested prediction or a causal claim.**


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.